In [ ]:

from langchain_openai import OpenAIEmbeddings
import dotenv 
from langchain_pinecone import PineconeVectorStore


dotenv.load_dotenv()
index_name = "tax-index"
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

C:\Users\soobi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o')

In [3]:
from typing import Annotated
from typing_extensions import List, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from langchain_core.documents import Document
import operator

class AgentState(TypedDict) :
    query:str
    context: List
    answer:str

In [4]:
from langgraph.graph import StateGraph


graph_builder = StateGraph(AgentState)

In [5]:
def retrieve(state: AgentState) -> AgentState:
 
    query = state['query']
    docs = retriever.invoke(query)
    return {'context': docs}

In [6]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client
from typing import Literal

client = Client()
doc_relevance_prompt = client.pull_prompt("langchain-ai/rag-document-relevance")

def check_doc_relevance(state: AgentState) -> Literal['relevant', 'irrelevant']:
   
    query = state['query']
    context = state['context']
    context_text = "\n\n".join([doc.page_content for doc in context])
    doc_relevance_chain = doc_relevance_prompt | llm
    response = doc_relevance_chain.invoke({'question': query, 'documents': context_text})

    print(f'response: {response["Score"]}')
    if response['Score'] == 1:
        # 2.3장과 다르게 `relevant`와 `irrelevant`를 반환합니다
        # node를 직접 지정하는 것보다 실제 판단 결과를 리턴하면서 해당 node의 재사용성을 높일 수 있습니다.
        return 'relevant'
    
    return 'irrelevant'

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


template = """당신은 질문-답변 과제를 수행하는 인공지능 어시스턴트입니다. 
주어진 문맥(Context)을 사용하여 질문(Question)에 답하세요. 
답을 모른다면 모른다고 답변하세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:"""

# 직접 프롬프트 객체 생성
generate_prompt = ChatPromptTemplate.from_template(template)
generate_llm = ChatOpenAI(model="gpt-4o")

def generate(state: AgentState) -> AgentState:
    
    context = state['context']
    query = state['query']
    
    
    rag_chain = generate_prompt | generate_llm

    response = rag_chain.invoke({'question': query, 'context': context})
    

    return {'answer': response.content}

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

dictionary = ['사람과 관련된 표현 -> 거주자']

rewrite_prompt = PromptTemplate.from_template(f"""
사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요 
사전: {dictionary}                                           
질문: {{query}}
""")

def rewrite(state: AgentState) -> AgentState:
    query = state['query']
    rewrite_chain = rewrite_prompt | llm | StrOutputParser()

    response = rewrite_chain.invoke({'query': query})
    return {'query': response}


In [9]:
# set the LANGCHAIN_API_KEY environment variable (create key in settings)
from langchain_core.output_parsers import StrOutputParser

hallucination_prompt = PromptTemplate.from_template("""
You are a teacher tasked with evaluating whether a student's answer is based on documents or not,
Given documents, which are excerpts from income tax law, and a student's answer;
If the student's answer is based on documents, respond with "not hallucinated",
If the student's answer is not based on documents, respond with "hallucinated".

documents: {documents}
student_answer: {student_answer}
""")

hallucination_llm = ChatOpenAI(model='gpt-4o', temperature=0)

def check_hallucination(state: AgentState) -> Literal['hallucinated', 'not hallucinated']:
    answer = state['answer']
    context = state['context']
    context = [doc.page_content for doc in context]
    hallucination_chain = hallucination_prompt | hallucination_llm | StrOutputParser()
    response = hallucination_chain.invoke({'student_answer': answer, 'documents': context})

    return response

In [10]:
from langsmith import Client
from typing import Literal


# LangChain 허브에서 유용성 프롬프트를 가져옵니다
client = Client()
helpfulness_prompt = client.pull_prompt("langchain-ai/rag-answer-helpfulness")

def check_helpfulness_grader(state: AgentState) -> str:
    
    # state에서 질문과 답변을 추출합니다
    query = state['query']
    answer = state['answer']

    # 답변의 유용성을 평가하기 위한 체인을 생성합니다
    helpfulness_chain = helpfulness_prompt | llm
    
    # 질문과 답변으로 체인을 호출합니다
    response = helpfulness_chain.invoke({'question': query, 'student_answer': answer})

    # 점수가 1이면 'helpful'을 반환하고, 그렇지 않으면 'unhelpful'을 반환합니다
    if response['Score'] == 1:
        return 'helpful'
    
    return 'unhelpful'


def check_helpfulness(state: AgentState) -> AgentState:
    """
    유용성을 확인하는 자리 표시자 함수입니다. 
    graph에서 conditional_edge를 연속으로 사용하지 않고 node를 추가해
    가독성을 높이기 위해 사용합니다

    Args:
        state (AgentState): 에이전트의 현재 state.

    Returns:
        AgentState: 변경되지 않은 state를 반환합니다.
    """
    # 이 함수는 현재 아무 작업도 수행하지 않으며 state를 그대로 반환합니다
    return state

In [11]:
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate',generate)
graph_builder.add_node('rewrite', rewrite)
graph_builder.add_node('check_helpfulness', check_helpfulness)


In [12]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_conditional_edges('retrieve', check_doc_relevance, 
{
    'relevant': 'generate',
    'irrelevant': END
}
 )
graph_builder.add_conditional_edges('generate',check_hallucination, {
    'not hallucinated': 'check_helpfulness',
    'hallucinated': 'generate'
})
graph_builder.add_conditional_edges('check_helpfulness', check_helpfulness_grader, {
    'helpful' : END,
    'unhelpful': 'rewrite'
})
graph_builder.add_edge('rewrite', 'retrieve')

In [13]:
graph = graph_builder.compile()

In [15]:
initial_state = {'query': '연봉 5천만원인 거주자가 납부할 소득세는 얼만가요?'}
graph.invoke(initial_state)

response: 1


{'query': '연봉 5천만원인 거주자가 납부할 소득세는 얼만가요?',
 'context': [Document(id='5c70e245-c348-4e94-9e1d-494a929d69ad', metadata={'source': 'output_tax_income.txt'}, page_content='[p. 105]\n    * 바. 「국민연금법」에 따라 받는 반환일시금(사망으로 받는 것만 해당한다) 및 사망일시금\n    * 사. 「공무원연금법」, 「공무원 재해보상법」, 「군인연금법」, 「군인 재해보상법」, 「사립학교교직원 연금법」또는「별정우체국법」에 따라 받는 공무상요양비·요양급여ㆍ장해일시금ㆍ비공무상 장해일시금ㆍ비직무상 장해일시금ㆍ장애보상금ㆍ사망조위금ㆍ사망보상금ㆍ유족일시금·퇴직유족일시금ㆍ유족연금일시금·퇴직유족연금일시금ㆍ퇴역유족연금일시금ㆍ순직유족연금일시금ㆍ유족연금부가금·퇴직유족연금부가금ㆍ퇴역유족연금부가금ㆍ유족연금특별부가금·퇴직유족연금특별부가금ㆍ퇴역유족연금특별부가금ㆍ순직유족보상금ㆍ직무상유족보상금ㆍ위험직무순직유족보상금·재해부조금·재난부조금 또는 신체·정신상의 장해·질병으로 인한 휴직기간에 받는 급여\n    * 아. 대통령령으로 정하는 학자금\n    * 자. 대통령령으로 정하는 실비변상적(實費辨償的) 성질의 급여\n    * 차. 외국정부(외국의 지방자치단체와 연방국가인 외국의 지방정부를 포함한다. 이하 같다) 또는 대통령령으로 정하는 국제기관에서 근무하는 사람으로서 대통령령으로 정하는 사람이 받는 급여. 다만, 그 외국정부가 그 나라에서 근무하는 우리나라 공무원의 급여에 대하여 소득세를 과세하지 아니하는 경우만 해당한다.\n법제처\n106\n국가법령정보센터\n--- [Section: Page 121-137] ---\n```markdown\n[p. 121]\n소득세법\n제3절 세액의 계산 <개정 2009. 12. 31.>\n제1관 세율 <개정 2009. 12. 31.>\n제55조(세율) ① 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세